In [79]:
!pip install rdkit

In [80]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw

##Question 1

In [81]:
reactions = ["CC(=O)O.CC>[H+].[Cl-]>CC(=O)OCC.O","C=C.[H][H]>[Pd]>CC", "c1ccccc1.O=[N+]([O-])O>>c1ccccc1[N+](=O)[O-].O"]

def parse(rxn):
    fields = rxn.split('>')
    if len(fields) != 3:
        raise ValueError(f"Invalid reaction string: {rxn}. Expected 2 '>' separators.")

    keys = ['reactants', 'reagents', 'products']
    result = {}

    for key, field_content in zip(keys, fields):
        molecules = [m for m in field_content.split('.') if m]
        result[key] = molecules

    return result

for r in reactions:
    parsed = parse(r)
    print(f"Reaction string: {r}")
    for field in ['reactants', 'reagents', 'products']:
        count = len(parsed[field])
        print(f"  {field.capitalize()}: {count} molecule(s) -> {parsed[field]}")
    print()


Reaction string: CC(=O)O.CC>[H+].[Cl-]>CC(=O)OCC.O
  Reactants: 2 molecule(s) -> ['CC(=O)O', 'CC']
  Reagents: 2 molecule(s) -> ['[H+]', '[Cl-]']
  Products: 2 molecule(s) -> ['CC(=O)OCC', 'O']

Reaction string: C=C.[H][H]>[Pd]>CC
  Reactants: 2 molecule(s) -> ['C=C', '[H][H]']
  Reagents: 1 molecule(s) -> ['[Pd]']
  Products: 1 molecule(s) -> ['CC']

Reaction string: c1ccccc1.O=[N+]([O-])O>>c1ccccc1[N+](=O)[O-].O
  Reactants: 2 molecule(s) -> ['c1ccccc1', 'O=[N+]([O-])O']
  Reagents: 0 molecule(s) -> []
  Products: 2 molecule(s) -> ['c1ccccc1[N+](=O)[O-]', 'O']



The first reaction is the esterification; an empty reagent field in the third reaction means no specific catalyst or solvent was specified.

##Question 2

In [87]:
import numpy as np

# Columns: C2H6, O2, CO2, H2O; Rows: C, H, O
E = np.array([
    [2, 0, -1, 0],
    [6, 0, 0, -2],
    [0, 2, -2, -1]
], dtype=float)

U, S, Vh = np.linalg.svd(E)
singular_vals = S
print("Singular vals:", singular_vals)


rank = np.linalg.matrix_rank(E)
nullity = E.shape[1] - rank
print(f"Matrix Rank: {rank}")
print(f"Nullity: {nullity}")

null_vector = Vh[-1, :]

coeff_scaled_1 = null_vector / null_vector[0]
print(f"Coefficients scaled to C2H6=1: {coeff_scaled_1}")

x = coeff_scaled_1 * 2
print(f"Smallest whole-number coefficients: {x}")

print(f"Verification E @ x: {verification}")
print()
if np.allclose(E @ x, 0):
    print(f"Verification E @ x = 0: {np.allclose(E @ x, 0)}")
    print("E @ x is approximately zero.")
else:
    print(f"Verification E @ x = 0: {np.allclose(E @ x, 0)}")
    print("E @ x is not approximately zero.")

Singular vals: [6.62561256 3.00720721 1.02857332]
Matrix Rank: 3
Nullity: 1
Coefficients scaled to C2H6=1: [1.  3.5 2.  3. ]
Smallest whole-number coefficients: [2. 7. 4. 6.]
Verification E @ x: [-8.88178420e-16 -3.55271368e-15 -1.77635684e-15]

Verification E @ x = 0: True
E @ x is approximately zero.


A null space dimension of one means the reaction has a unique stoichiometry  for the given species.

##Question 3

In [84]:
import numpy as np

A = np.zeros((4,4))
for i in range(3):
    A[i, i+1] = 1
    A[i+1, i] = 1

degrees = np.sum(A, axis=0)

eigenvalues = np.linalg.eigvals(A)
eigenvalues = np.sort(eigenvalues)[::-1]

e_pi_coeff = 2 * (eigenvalues[0] + eigenvalues[1])
deloc_energy_coeff = e_pi_coeff - 4

print(f"The four eigenvalues: {eigenvalues}")
print(f"Degree of every atom: {degrees}")
print(f"E_pi: 4*alpha + {e_pi_coeff:.4f}*beta")
print(f"Delocalization Energy: {deloc_energy_coeff:.4f}*beta")

The four eigenvalues: [ 1.61803399  0.61803399 -0.61803399 -1.61803399]
Degree of every atom: [1. 2. 2. 1.]
E_pi: 4*alpha + 4.4721*beta
Delocalization Energy: 0.4721*beta


Butadiene's delocalization energy 0.4721*beta is much smaller than benzene's 2 * beta,
indicating that benzene's cyclic aromaticity provides significantly greater resonance stability.

##Question 4

In [85]:
import numpy as np

A6 = np.zeros((6, 6))
for i in range(6):
    A6[i, (i + 1) % 6] = 1.0
    A6[(i + 1) % 6, i] = 1.0

A_tilde = A6 + np.eye(6)
D_tilde_inv = np.diag(1.0 / np.sum(A_tilde, axis=1))
P = D_tilde_inv @ A_tilde

evals = np.linalg.eigvals(P)
moduli = np.sort(np.abs(evals))[::-1]
mu = moduli[1]

rng = np.random.default_rng(1)
H = rng.normal(size=(6, 3))
mean_row = np.mean(H, axis=0)

print(f"Eigenvalue moduli of P: {moduli}")
print(f"Smoothing rate (mu): {mu:.4f}\n")

ks = [0, 1, 2, 4, 8, 16]
H_k = H.copy()

for k in range(max(ks) + 1):
    if k in ks:
        deviation = np.max(np.linalg.norm(H_k - mean_row, axis=1))
        print(f"k={k:2d} | Max deviation from mean: {deviation:.6f}")
    H_k = P @ H_k


Eigenvalue moduli of P: [1.00000000e+00 6.66666667e-01 6.66666667e-01 3.33333333e-01
 7.91848035e-17 3.92523115e-17]
Smoothing rate (mu): 0.6667

k= 0 | Max deviation from mean: 1.241388
k= 1 | Max deviation from mean: 0.536825
k= 2 | Max deviation from mean: 0.345910
k= 4 | Max deviation from mean: 0.154873
k= 8 | Max deviation from mean: 0.030668
k=16 | Max deviation from mean: 0.001197


Implication: With mu around 0.33, features converge rapidly; 4-8 layers are plenty to smooth features for a molecule of ~12 heavy atoms.

##Question 5


In [86]:
import numpy as np

X = np.array([[78.1, 2.3],
              [92.1, 1.7],
              [106.2, 2.8],
              [120.2, 2.0],
              [134.2, 3.1],
              [148.2, 2.4]])

def run_pca(data, name):
    N = data.shape[0]
    Xc = data - np.mean(data, axis=0)

    Cov = (Xc.T @ Xc) / N

    evals, evecs = np.linalg.eigh(Cov)

    idx = np.argsort(evals)[::-1]
    evals = evals[idx]
    evecs = evecs[:, idx]

    for i in range(evecs.shape[1]):
        if evecs[np.argmax(np.abs(evecs[:, i])), i] < 0:
            evecs[:, i] *= -1

    var_exp = evals / np.sum(evals)

    print(f"{name} PCA")
    print(f"Covariance Matrix:\n{Cov}")
    print(f"Eigenvalues: {evals}")
    print(f"Fraction of Variance Explained: {var_exp}")
    print(f"First Principal Direction: {evecs[:, 0]}")
    print()
    return evecs[:, 0]

run_pca(X, "Centered")
run_pca(X / np.std(X, axis=0), "Standardized")

Centered PCA
Covariance Matrix:
[[5.73535556e+02 4.56277778e+00]
 [4.56277778e+00 2.18055556e-01]]
Eigenvalues: [5.73571866e+02 1.81744746e-01]
Fraction of Variance Explained: [9.99683236e-01 3.16764448e-04]
First Principal Direction: [0.99996834 0.0079578 ]

Standardized PCA
Covariance Matrix:
[[1.         0.40800508]
 [0.40800508 1.        ]]
Eigenvalues: [1.40800508 0.59199492]
Fraction of Variance Explained: [0.70400254 0.29599746]
First Principal Direction: [0.70710678 0.70710678]



array([0.70710678, 0.70710678])

The two analyses disagree because molar mass has a much larger numerical range than dipole moment; I would report the standardized analysis to ensure both features contribute equally.